In [ ]:
import sys
sys.path.insert(0, 'include')

from include.actor_system import ActorSystem
from threading import Event
from data.lte_system_info import LTEParams
import numpy as np
from include.buffer_manager_actor import BufferManagerActor
from include.flow_graph import FlowGraph, SlotPool, ResizableSlotPool, FunctionNode
from include.slot_type import CellSearchSlot, MIBSlot, SIB1Slot
from include.cell_search import PSSDetection, SSSDetection
from include.msg_type import PipelineDone
from include.cell_search_actor import CellSearchActor
from include.crs_estimate import CRSChannelEstimation
from include.mib_decode import PBCHDecoding, BCHDecoding
from include.mib_decode_actor import MIBDecodeActor
from include.sib1_decode_control import PCFICHDecoding, PDCCHDecoding
from include.sib1_decode_data import PDSCHDecoding, DLSCHDecoding
from include.sib1_decode_actor import SIB1DecodeActor
from include.controller_actor import ControllerActor
import time

In [2]:
rxf = np.load('data/rx_preprocessed.npy')
Fs = float(np.load('data/Fs.npy'))
params = LTEParams(Fs=Fs)

In [ ]:
def run_full_system(n_workers, cs_cc, mib_cc, sib1_cc, ingest_delay, n_slot, buffer_size,
                     batch_size, max_tracking=10, sib1_target=50, verbose=True):

    system = ActorSystem()
    done = Event()

    # ---- Buffer Manager ----
    bm = BufferManagerActor(system, rxf,
                            buffer_size=buffer_size,
                            batch_size=batch_size,
                            ingest_delay=ingest_delay)

    # ---- One shared FlowGraph (one thread pool) ----
    graph = FlowGraph(num_workers=n_workers)

    # ---- Cell Search pipeline ----
    cs_pool = ResizableSlotPool(n_slot, lambda: CellSearchSlot(params.N_subframe))
    pss_func = PSSDetection(params, peak_ratio=5.0)
    sss_func = SSSDetection(params, peak_ratio=8.0)

    pss_node = FunctionNode('pss', cs_pool.make_stage(pss_func), concurrency=cs_cc)
    sss_node = FunctionNode('sss', cs_pool.make_stage(sss_func), concurrency=cs_cc,
                            done_callback=lambda token: system.send_message(
                                'cell_search', PipelineDone(slot=token.slot, tag=token.tag)))
    graph.add_edge(pss_node, sss_node)

    cs = CellSearchActor(system, bm.buf, cs_pool, graph, params, [pss_node, sss_node])

    # ---- MIB pipeline (same graph, different chain) ----
    mib_pool = SlotPool(n_slot, lambda: MIBSlot(params.pbch_len))
    crs_mib   = CRSChannelEstimation(params)
    pbch_func = PBCHDecoding()
    bch_func  = BCHDecoding()

    crs_mib_node = FunctionNode('mib_crs',  mib_pool.make_stage(crs_mib),  concurrency=mib_cc)
    pbch_node    = FunctionNode('mib_pbch', mib_pool.make_stage(pbch_func), concurrency=mib_cc)
    bch_node     = FunctionNode('mib_bch',  mib_pool.make_stage(bch_func),  concurrency=mib_cc,
                                done_callback=lambda token: system.send_message(
                                    'mib_decode', PipelineDone(slot=token.slot, tag=token.tag)))
    graph.add_edge(crs_mib_node, pbch_node)
    graph.add_edge(pbch_node, bch_node)

    mib = MIBDecodeActor(system, bm.buf, mib_pool, graph,
                         [crs_mib_node, pbch_node, bch_node])

    # ---- SIB1 pipeline (same graph, different chain) ----
    sib1_pool = SlotPool(n_slot, lambda: SIB1Slot(params.N_subframe))
    crs_sib1    = CRSChannelEstimation(params)
    pcfich_func = PCFICHDecoding()
    pdcch_func  = PDCCHDecoding()
    pdsch_func  = PDSCHDecoding()
    dlsch_func  = DLSCHDecoding()

    crs_sib1_node = FunctionNode('sib1_crs',    sib1_pool.make_stage(crs_sib1),    concurrency=sib1_cc)
    pcfich_node   = FunctionNode('sib1_pcfich', sib1_pool.make_stage(pcfich_func), concurrency=sib1_cc)
    pdcch_node    = FunctionNode('sib1_pdcch',  sib1_pool.make_stage(pdcch_func),  concurrency=sib1_cc)
    pdsch_node    = FunctionNode('sib1_pdsch',  sib1_pool.make_stage(pdsch_func),  concurrency=sib1_cc)
    dlsch_node    = FunctionNode('sib1_dlsch',  sib1_pool.make_stage(dlsch_func),  concurrency=sib1_cc,
                                 done_callback=lambda token: system.send_message(
                                     'sib1_decode', PipelineDone(slot=token.slot, tag=token.tag)))
    graph.add_edge(crs_sib1_node, pcfich_node)
    graph.add_edge(pcfich_node, pdcch_node)
    graph.add_edge(pdcch_node, pdsch_node)
    graph.add_edge(pdsch_node, dlsch_node)

    sib1 = SIB1DecodeActor(system, bm.buf, sib1_pool, graph,
                           [crs_sib1_node, pcfich_node, pdcch_node, pdsch_node, dlsch_node])

    # ---- Controller ----
    ctrl = ControllerActor(system, params, max_tracking=max_tracking,
                           sib1_target=sib1_target, done_event=done)

    # ---- Register all actors ----
    system.create_actor(bm)
    system.create_actor(cs)
    system.create_actor(mib)
    system.create_actor(sib1)
    system.create_actor(ctrl)

    # ---- One message starts everything ----
    t0 = time.perf_counter()
    system.send_message('buffer_manager', 'start')

    done.wait(timeout=120)
    elapsed = time.perf_counter() - t0
    graph.shutdown()

    if verbose:
        n_sib1 = len(ctrl.sib1_results)
        print(f'=== Full System Results ===')
        print(f'Time: {elapsed:.3f}s')
        print(f'Cell: N_id={ctrl.N_id}, BW={ctrl.dl_bw}, n_ant={ctrl.n_ant}')
        print(f'SIB1: {n_sib1}/{sib1_target} decoded')
        print(f'Unit time: {elapsed/max(n_sib1,1)*1000:.2f} ms/chunk')
        print()
        for r in ctrl.sib1_results[:5]:
            print(f'  {r["frame_tag"]}: SFN={r["sfn"]}, {len(r["sib1_bytes"])} bytes')
        print(f'\nLog ({len(ctrl.log)} entries):')
        for entry in ctrl.log[:10]:
            print(f'  {entry}')
        if len(ctrl.log) > 10:
            print(f'  ... ({len(ctrl.log) - 10} more)')
        for entry in ctrl.log[-10:]:
            print(entry)

    return elapsed, ctrl

#### Performance: the total execution time < 1 second
1. baseline: infinite buffer size + all data (infinite data ingestion speed) + serial concurrency + infinite tracking

In [10]:
elapsed, ctrl = run_full_system(n_workers=6, cs_cc=1, mib_cc=1, sib1_cc=1, n_slot=4, ingest_delay=0.0, buffer_size = len(rxf),
                     batch_size = len(rxf), max_tracking=100)

=== Full System Results ===
Time: 0.280s
Cell: N_id=380, BW=50, n_ant=2
SIB1: 50/50 decoded
Unit time: 5.61 ms/chunk

  frame_2: SFN=314, 22 bytes
  frame_3: SFN=316, 22 bytes
  frame_4: SFN=318, 22 bytes
  frame_5: SFN=320, 22 bytes
  frame_6: SFN=322, 22 bytes

Log (510 entries):
  IDLE → CELL_SEARCH
  CELL_SEARCH: dispatched 5 chunks
  CELL_SEARCH → TRACKING: N_id=380, pss_global=36043, frame_start=29387
  CONSUME: cell_search_align, amount=29387, cursor: 0 → 29387
  TRACKING: mib_sent frame_1
  TRACKING: mib_done frame_1, SFN=313, BW=50, n_ant=2
  TRACKING: SFN=313 odd, skip SIB1
  CONSUME: after_frame_1, amount=153600, cursor: 29387 → 182987
  TRACKING: cleanup 1 frames: frame_1(SFN=313)
  TRACKING: pss_sent frame_2
  ... (500 more)
TRACKING: pss_sent frame_148
TRACKING: sib1_done frame_49, SFN=408
TRACKING: cleanup 1 frames: frame_49(SFN=408)
TRACKING: pss_sent frame_149
TRACKING: sib1_done frame_50, SFN=410
TRACKING: cleanup 1 frames: frame_50(SFN=410)
TRACKING: pss_sent frame_1

good for unit time less than 20 ms/chunk as 1 sec data long/50 sib1 = 20 ms/chunk

2. infinite buffer size + all data (infinite data ingestion speed) + **good parallel concurrency settings (from test)** + infinite tracking

In [12]:
elapsed, ctrl = run_full_system(n_workers=6, cs_cc=6, mib_cc=1, sib1_cc=6, n_slot=4, ingest_delay=0.0, buffer_size = len(rxf),
                     batch_size = len(rxf), max_tracking=100)

=== Full System Results ===
Time: 0.245s
Cell: N_id=380, BW=50, n_ant=2
SIB1: 50/50 decoded
Unit time: 4.89 ms/chunk

  frame_3: SFN=316, 22 bytes
  frame_2: SFN=314, 22 bytes
  frame_4: SFN=318, 22 bytes
  frame_5: SFN=320, 22 bytes
  frame_6: SFN=322, 22 bytes

Log (493 entries):
  IDLE → CELL_SEARCH
  CELL_SEARCH: dispatched 5 chunks
  CELL_SEARCH → TRACKING: N_id=380, pss_global=36043, frame_start=29387
  CONSUME: cell_search_align, amount=29387, cursor: 0 → 29387
  TRACKING: mib_sent frame_1
  TRACKING: mib_done frame_1, SFN=313, BW=50, n_ant=2
  TRACKING: SFN=313 odd, skip SIB1
  CONSUME: after_frame_1, amount=153600, cursor: 29387 → 182987
  TRACKING: cleanup 1 frames: frame_1(SFN=313)
  TRACKING: pss_sent frame_2
  ... (483 more)
TRACKING: cleanup 1 frames: frame_48(SFN=406)
TRACKING: pss_sent frame_148
TRACKING: sib1_done frame_49, SFN=408
TRACKING: cleanup 1 frames: frame_49(SFN=408)
TRACKING: pss_sent frame_149
TRACKING: sib1_done frame_51, SFN=412
TRACKING: sib1_done frame_

3. infinite buffer size + **normal data ingestion speed** + **good parallel concurrency settings (from test)** + infinite tracking

In [6]:
elapsed, ctrl = run_full_system(n_workers=6, cs_cc=6, mib_cc=1, sib1_cc=6, n_slot=4, ingest_delay=0.001, buffer_size = len(rxf),
                     batch_size = params.N_subframe, max_tracking=100)

=== Full System Results ===
Time: 1.170s
Cell: N_id=380, BW=50, n_ant=2
SIB1: 50/50 decoded
Unit time: 23.39 ms/chunk

  frame_2: SFN=314, 22 bytes
  frame_3: SFN=316, 22 bytes
  frame_4: SFN=318, 22 bytes
  frame_5: SFN=320, 22 bytes
  frame_6: SFN=322, 22 bytes

Log (509 entries):
  IDLE → CELL_SEARCH
  CELL_SEARCH: dispatched 5 chunks
  CELL_SEARCH → TRACKING: N_id=380, pss_global=36043, frame_start=29387
  CONSUME: cell_search_align, amount=29387, cursor: 0 → 29387
  TRACKING: mib_sent frame_1
  TRACKING: mib_done frame_1, SFN=313, BW=50, n_ant=2
  TRACKING: SFN=313 odd, skip SIB1
  CONSUME: after_frame_1, amount=153600, cursor: 29387 → 182987
  TRACKING: cleanup 1 frames: frame_1(SFN=313)
  TRACKING: pss_sent frame_2
  ... (499 more)
TRACKING: cleanup 1 frames: frame_50(SFN=410)
TRACKING: pss_sent frame_150
TRACKING: pss_done frame_51, pos=15242419
TRACKING: mib_sent frame_51
TRACKING: mib_done frame_51, SFN=412, BW=50, n_ant=2
TRACKING: sib1_sent frame_51, SFN=412
CONSUME: after_

This is as expected since in the setting of buffer manager, once data is not sufficient to read, wait for time. This is caused by infinite tracking in the controller, that is, send all possible pss immediately in the tracking mode.

4. infinite buffer size + **normal data ingestion speed** + **good parallel concurrency settings (from test)** + **finite tracking**

In [7]:
elapsed, ctrl = run_full_system(n_workers=6, cs_cc=6, mib_cc=1, sib1_cc=6, n_slot=4, ingest_delay=0.001, buffer_size = len(rxf),
                     batch_size = params.N_subframe, max_tracking=10)

=== Full System Results ===
Time: 1.082s
Cell: N_id=380, BW=50, n_ant=2
SIB1: 50/50 decoded
Unit time: 21.64 ms/chunk

  frame_2: SFN=314, 22 bytes
  frame_3: SFN=316, 22 bytes
  frame_4: SFN=318, 22 bytes
  frame_5: SFN=320, 22 bytes
  frame_6: SFN=322, 22 bytes

Log (419 entries):
  IDLE → CELL_SEARCH
  CELL_SEARCH: dispatched 5 chunks
  CELL_SEARCH → TRACKING: N_id=380, pss_global=36043, frame_start=29387
  CONSUME: cell_search_align, amount=29387, cursor: 0 → 29387
  TRACKING: mib_sent frame_1
  TRACKING: mib_done frame_1, SFN=313, BW=50, n_ant=2
  TRACKING: SFN=313 odd, skip SIB1
  CONSUME: after_frame_1, amount=153600, cursor: 29387 → 182987
  TRACKING: cleanup 1 frames: frame_1(SFN=313)
  TRACKING: pss_sent frame_2
  ... (409 more)
TRACKING: pss_done frame_51, pos=15242422
TRACKING: mib_sent frame_51
TRACKING: pss_done frame_52, pos=15549622
TRACKING: mib_sent frame_52
TRACKING: mib_done frame_51, SFN=404, BW=50, n_ant=2
TRACKING: sib1_sent frame_51, SFN=404
CONSUME: after_frame

5. **finite buffer size (real)** + **normal data ingestion speed** + **good parallel concurrency settings (from test)** + **finite tracking**

In [8]:
elapsed, ctrl = run_full_system(n_workers=6, cs_cc=6, mib_cc=1, sib1_cc=6, n_slot=4, ingest_delay=0.001, buffer_size = 5*params.N_frame,
                     batch_size = params.N_subframe, max_tracking=10)

=== Full System Results ===
Time: 1.082s
Cell: N_id=380, BW=50, n_ant=2
SIB1: 50/50 decoded
Unit time: 21.64 ms/chunk

  frame_2: SFN=314, 22 bytes
  frame_3: SFN=316, 22 bytes
  frame_4: SFN=318, 22 bytes
  frame_5: SFN=320, 22 bytes
  frame_6: SFN=322, 22 bytes

Log (419 entries):
  IDLE → CELL_SEARCH
  CELL_SEARCH: dispatched 5 chunks
  CELL_SEARCH → TRACKING: N_id=380, pss_global=36043, frame_start=29387
  CONSUME: cell_search_align, amount=29387, cursor: 0 → 29387
  TRACKING: mib_sent frame_1
  TRACKING: mib_done frame_1, SFN=313, BW=50, n_ant=2
  TRACKING: SFN=313 odd, skip SIB1
  CONSUME: after_frame_1, amount=153600, cursor: 29387 → 182987
  TRACKING: cleanup 1 frames: frame_1(SFN=313)
  TRACKING: pss_sent frame_2
  ... (409 more)
TRACKING: pss_done frame_51, pos=15242422
TRACKING: mib_sent frame_51
TRACKING: pss_done frame_52, pos=15549622
TRACKING: mib_sent frame_52
TRACKING: mib_done frame_51, SFN=404, BW=50, n_ant=2
TRACKING: sib1_sent frame_51, SFN=404
CONSUME: after_frame

6. **finite buffer size (real)** + **normal data ingestion speed** + **good parallel concurrency settings (from test)** + **small tracking**

In [9]:
elapsed, ctrl = run_full_system(n_workers=6, cs_cc=6, mib_cc=1, sib1_cc=6, n_slot=4, ingest_delay=0.001, buffer_size = 5*params.N_frame,
                     batch_size = params.N_subframe, max_tracking=5)

=== Full System Results ===
Time: 0.975s
Cell: N_id=380, BW=50, n_ant=2
SIB1: 50/50 decoded
Unit time: 19.50 ms/chunk

  frame_2: SFN=314, 22 bytes
  frame_3: SFN=316, 22 bytes
  frame_4: SFN=318, 22 bytes
  frame_5: SFN=320, 22 bytes
  frame_6: SFN=322, 22 bytes

Log (414 entries):
  IDLE → CELL_SEARCH
  CELL_SEARCH: dispatched 5 chunks
  CELL_SEARCH → TRACKING: N_id=380, pss_global=36043, frame_start=29387
  CONSUME: cell_search_align, amount=29387, cursor: 0 → 29387
  TRACKING: mib_sent frame_1
  TRACKING: mib_done frame_1, SFN=313, BW=50, n_ant=2
  TRACKING: SFN=313 odd, skip SIB1
  CONSUME: after_frame_1, amount=153600, cursor: 29387 → 182987
  TRACKING: cleanup 1 frames: frame_1(SFN=313)
  TRACKING: pss_sent frame_2
  ... (404 more)
TRACKING: pss_done frame_52, pos=15549712
TRACKING: mib_sent frame_52
TRACKING: pss_done frame_51, pos=15242471
TRACKING: mib_sent frame_51
TRACKING: mib_done frame_51, SFN=394, BW=50, n_ant=2
TRACKING: sib1_sent frame_51, SFN=394
CONSUME: after_frame

That is it. Reduce tracking size in controller can release some executing resource for sib1. And the final time is reduced to less than 1 second.